# 7-Qubit Superconducting Quantum Processor - Planar CAD Layout
### Reference: Quantum Computer Design Series (Part 1: Quantum Chip) - Simon Quantum Hardware
* **Toolchain:** Qiskit Metal 0.1.5 | Python 3.10 | KLayout
* **Chip Canvas:** 9 mm × 7 mm high-resistivity silicon (center at `[0 mm, 1 mm]`)
* **Components:** 7 Transmons (`Q1`–`Q7` with `Q4` TransmonPocketCL) + 7 Launchpads (`LP1`–`LP7`) + LineTee Couplers (`Lt1`–`Lt4`) + RouteMixed Meanders (`RM1`–`RM4`) + RouteStraight Bridges (`RS1`–`RS3`)


In [ ]:
# Cell 1: Environment & Library Imports
%reload_ext autoreload
%autoreload 2

from PIL import Image
import numpy as np
from collections import OrderedDict

import qiskit_metal as metal
from qiskit_metal import designs, draw, MetalGUI, Dict, Headings
from qiskit_metal.qlibrary.qubits.transmon_pocket_6 import TransmonPocket6
from qiskit_metal.qlibrary.qubits.transmon_pocket_cl import TransmonPocketCL
from qiskit_metal.qlibrary.couplers.line_tee import LineTee
from qiskit_metal.qlibrary.couplers.coupled_line_tee import CoupledLineTee
from qiskit_metal.qlibrary.terminations.launchpad_wb import LaunchpadWirebond
from qiskit_metal.qlibrary.tlines.mixed_path import RouteMixed
from qiskit_metal.qlibrary.tlines.straight_path import RouteStraight
from qiskit_metal.qlibrary.core import QComponent
import pyEPR as epr

print("Imported Qiskit Metal and microwave design libraries successfully!")


In [ ]:
# Cell 2: Initialize Planar Chip Canvas (9mm x 7mm)
design = designs.DesignPlanar({}, True)
design.chips.main.size['size_x'] = '9mm'
design.chips.main.size['size_y'] = '7mm'
design.chips.main.size['center_x'] = '0mm'
design.chips.main.size['center_y'] = '1mm'

# Launch Native Desktop Qiskit Metal Qt GUI
gui = MetalGUI(design)
gui.autoscale()
gui.rebuild()
print("Initialized 9mm x 7mm chip canvas in Qiskit Metal GUI!")


In [ ]:
# Cell 3: Place 7 Transmons (Q1-Q7)
# Standard Connection Pad template for outer transmons (a and b pads)
outer_pad_opts = {
    'cpw_extend': '100um',
    'cpw_gap': '6um',
    'cpw_width': '10um',
    'loc_H': '+1',
    'loc_W': '0',
    'pad_cpw_extent': '25um',
    'pad_cpw_shift': '0um',
    'pad_gap': '15um',
    'pad_height': '30um',
    'pad_width': '50um',
    'pocket_extent': '5um',
    'pocket_rise': '0um'
}

# Outer Transmons (Q1, Q2, Q3, Q5, Q6, Q7)
outer_qubits_config = {
    "Q1": Dict(pos_x="-2.5mm", pos_y="2.0mm", orientation="90",
               connection_pads=Dict(a=Dict(**outer_pad_opts, loc_H='+1'),
                                    b=Dict(**outer_pad_opts, loc_H='-1'))),
    "Q2": Dict(pos_x="0.0mm",  pos_y="2.0mm", orientation="90",
               connection_pads=Dict(a=Dict(**outer_pad_opts, loc_H='+1'),
                                    b=Dict(**outer_pad_opts, loc_H='-1'))),
    "Q3": Dict(pos_x="2.5mm",  pos_y="2.0mm", orientation="90",
               connection_pads=Dict(a=Dict(**outer_pad_opts, loc_H='+1'),
                                    b=Dict(**outer_pad_opts, loc_H='-1'))),
    "Q5": Dict(pos_x="-2.5mm", pos_y="0.0mm", orientation="90",
               connection_pads=Dict(a=Dict(**outer_pad_opts, loc_H='+1'),
                                    b=Dict(**outer_pad_opts, loc_H='-1'))),
    "Q6": Dict(pos_x="0.0mm",  pos_y="0.0mm", orientation="90",
               connection_pads=Dict(a=Dict(**outer_pad_opts, loc_H='+1'),
                                    b=Dict(**outer_pad_opts, loc_H='-1'))),
    "Q7": Dict(pos_x="2.5mm",  pos_y="0.0mm", orientation="90",
               connection_pads=Dict(a=Dict(**outer_pad_opts, loc_H='+1'),
                                    b=Dict(**outer_pad_opts, loc_H='-1'))),
}

for name, opts in outer_qubits_config.items():
    opts.update(Dict(
        pad_width="455um",
        pad_height="90um",
        pad_gap="30um",
        pocket_width="650um",
        pocket_height="650um"
    ))
    TransmonPocket6(design, name, options=opts)

# Center Transmon with Charge Line (Q4)
q4_opts = Dict(
    pos_x="0.0um",
    pos_y="870.0um",
    orientation="0.0",
    pad_width="455um",
    pad_height="90um",
    pad_gap="30um",
    pocket_width="650um",
    pocket_height="650um",
    make_CL=True,
    cl_gap="6um",
    cl_width="10um",
    cl_length="20um",
    cl_ground_gap="6um",
    cl_pocket_edge="50",
    cl_off_center="0um",
    connection_pads=Dict(
        a=Dict(loc_W=-1, loc_H=-1, pad_width='50um', pad_height='30um', cpw_width='10um', cpw_gap='6um'),
        b=Dict(loc_W=1,  loc_H=-1, pad_width='50um', pad_height='30um', cpw_width='10um', cpw_gap='6um')
    )
)
TransmonPocketCL(design, "Q4", options=q4_opts)

design.rebuild()
gui.autoscale()
print("Placed all 7 Transmons (Q1-Q7 with central Q4 TransmonPocketCL)!")


In [ ]:
# Cell 4: Place 7 Launchpads (LP1-LP7)
launchpads = {
    "LP1": Dict(pos_x="-3.5mm", pos_y="4.0mm", orientation="-90.0"),
    "LP2": Dict(pos_x="-1.2mm", pos_y="4.0mm", orientation="-90.0"),
    "LP3": Dict(pos_x="1.2mm",  pos_y="4.0mm", orientation="-90.0"),
    "LP4": Dict(pos_x="3.5mm",  pos_y="4.0mm", orientation="-90.0"),
    "LP5": Dict(pos_x="-2.5mm", pos_y="-2.0mm", orientation="90.0"),
    "LP6": Dict(pos_x="0.0mm",  pos_y="-2.0mm", orientation="90.0"),
    "LP7": Dict(pos_x="2.5mm",  pos_y="-2.0mm", orientation="90.0"),
}

for name, opts in launchpads.items():
    LaunchpadWirebond(design, name, options=opts)

design.rebuild()
gui.autoscale()
print("Placed all 7 Wirebond Launchpads (LP1-LP7)!")


In [ ]:
# Cell 5: Place LineTee Couplers (Lt1-Lt4)
line_tees = {
    "Lt1": Dict(pos_x="500.0um",  pos_y="1000.0um", orientation="0.0"),
    "Lt2": Dict(pos_x="-500.0um", pos_y="1000.0um", orientation="180.0"),
    "Lt3": Dict(pos_x="1500.0um", pos_y="1000.0um", orientation="0.0"),
    "Lt4": Dict(pos_x="-1500.0um", pos_y="1000.0um", orientation="180.0"),
}

for name, opts in line_tees.items():
    opts.update(Dict(
        prime_width="10um",
        prime_gap="6um",
        second_width="10um",
        second_gap="6um",
        t_length="50um"
    ))
    LineTee(design, name, options=opts)

design.rebuild()
gui.autoscale()
print("Placed LineTee Couplers (Lt1-Lt4)!")


In [ ]:
# Cell 6: Route Anchor-Guided Mixed Meanders (RouteMixed)
# RM1: Route from Lt1 second_end to Lt4 second_end
anchors4c = OrderedDict()
anchors4c[0] = np.array([0.5, 0.0])
anchors4c[1] = np.array([1.83, 0.0])
anchors4c[2] = np.array([1.83, 1.0])

anchors4cb = OrderedDict()
anchors4cb[0] = "S"
anchors4cb[1] = "M"
anchors4cb[2] = "S"
anchors4cb[3] = "S"

optionsR1 = {
    'pin_inputs': {
        'start_pin': {'component': 'Lt1', 'pin': 'second_end'},
        'end_pin': {'component': 'Lt4', 'pin': 'second_end'}
    },
    'lead': {'start_straight': '200um', 'end_straight': '200um'},
    'anchors': anchors4c,
    'between_anchors': anchors4cb,
    'fillet': '90um',
    'total_length': '10mm',
    'step_size': '0.1mm',
    'meander': {
        'spacing': '190um',
        'asymmetry': '0um'
    }
}
RouteMixed(design, 'RM1', optionsR1)

design.rebuild()
gui.autoscale()
print("Routed anchor-guided RouteMixed meander RM1 successfully!")


In [ ]:
# Cell 7: Route Straight Coupler Interconnects (RouteStraight)
straight_routes = [
    ("RS1", "Lt1", "prime_end", "Q1", "a"),
    ("RS2", "Lt1", "prime_start", "Q4", "a"),
    ("RS3", "Lt2", "prime_end", "Q4", "b"),
]

for name, comp_start, pin_start, comp_end, pin_end in straight_routes:
    rs_opts = Dict(
        pin_inputs=Dict(
            start_pin=Dict(component=comp_start, pin=pin_start),
            end_pin=Dict(component=comp_end, pin=pin_end)
        ),
        trace_gap='cpw_gap',
        type='CPW'
    )
    RouteStraight(design, name, options=rs_opts)

design.rebuild()
gui.autoscale()
print("Routed straight coupler interconnects RS1, RS2, RS3!")


In [ ]:
# Cell 8: Export Nanofabrication Mask (GDSII)
a_gds = design.renderers.gds
a_gds.export_to_gds("seven_qubit_chip_mask.gds")
print("Exported cleanroom mask to seven_qubit_chip_mask.gds for KLayout inspection!")
